# 05 — Where Black-Scholes fails

Black-Scholes is not wrong as *a formula*; the formula is exact given its
assumptions. It fails as a description of the world. The cleanest evidence is that
the one number it holds constant — volatility — is not constant anywhere:

* **across strikes** it smiles/smirks (fat tails, crash risk, leverage effect);
* **across maturities** it term-structures (event risk, mean reversion);
* **across time** it clusters (vol of vol), which is why a delta-hedged book is
  never actually riskless.

We use the sample chain to show the smile, show that the surface is not flat, and
show the PnL consequence for a hedger who believes the single number.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from optlab.pricing.black_scholes import bs_price, bs_greeks
from optlab.pricing.binomial import binomial_price
from optlab.pricing.monte_carlo import mc_price

from optlab.data.cleaning import clean_chain
from optlab.data.loaders import load_csv_chain
from optlab.hedging.simulate import simulate_delta_hedge
from optlab.surface.interpolate import VolSurface

In [ ]:
raw = load_csv_chain("data/sample/option_chain_sample.csv")
chain = clean_chain(raw, spot=raw.attrs["spot"], r=0.04, q=0.015)
surface = VolSurface.from_quotes(chain["strike"], chain["iv"], chain["T"], chain["forward"])

T = float(surface.maturities[1])
fwd = surface.forward(T)
strikes = np.linspace(80, 125, 25)
smile = surface.iv(strikes, T)
pd.DataFrame({"strike": strikes, "iv": smile}).assign(
    vol_points_vs_atm=lambda d: (d["iv"] - surface.iv(fwd, T)) * 100
).round(4).iloc[::4]

## 1. The smile is the market telling you the model is wrong

A single vol cannot price both the 90-strike and the 110-strike option. The market
quotes a *different* vol for each, and the shape is systematic: OTM puts trade
rich (crash protection), which is exactly the fat left tail GBM does not have.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for T_i in surface.maturities:
    ks = np.linspace(-0.25, 0.15, 80)
    ax.plot(ks, surface.iv(surface.forward(T_i) * np.exp(ks), T_i) * 100, label=f"T={T_i:.2f}")
ax.axhline(20, color="black", linestyle="--", linewidth=1, label="flat 20% (Black-Scholes)")
ax.set_xlabel("ln(K/F)")
ax.set_ylabel("implied vol (%)")
ax.set_title("Skew short-dated, flatter long-dated: nothing like a constant sigma")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2. Model risk shows up as PnL

Simulate a world where vol is 28% but hedge with 20% vols (think: a trader who
prices everything off one historical number). The hedge is delta-neutral every
day, yet the PnL distribution has a strong drift — the position is long gamma,
paid for by the option being underpriced relative to reality.

Flip it around: if you sell at 20% and realise 28%, you lose. Black-Scholes says
both should be zero expected PnL.

In [ ]:
rows = []
for quoted in [0.14, 0.20, 0.28]:
    res = simulate_delta_hedge(100, 100, 1.0, 0.04, quoted, 0.28, n_steps=252, n_paths=4000, seed=7)
    rows.append({"vol used to price/hedge": quoted, "vol realised": 0.28,
                 "mean PnL": res.mean, "std": res.std, "95% CI half-width": 1.96 * res.stderr})
pd.DataFrame(rows).round(4)

## 3. What the model is still good for

None of the above makes Black-Scholes useless. It is:

* the **quoting convention** — the market speaks in implied vols, and the formula
  is the translator between price and vol;
* the **risk language** — delta, gamma, vega and theta are defined off it, and the
  delta-hedged, gamma-rented PnL is what a vol trader actually manages;
* the **right answer in the right limit** — if volatility really were constant and
  you could trade continuously, the hedge would be exact.

What it is not is a *model of the world*. Everything past this notebook is the
industry's patch: fit the surface, enforce its arbitrage-free structure, hedge the
Greeks, and remember that the residual is a real PnL.

## Where to go next

* Fit SVI with a no-arbitrage penalty and compare the two fits (notebook 03).
* Add jumps or stochastic vol (Heston) and see the smile emerge endogenously.
* Use the real chain in `data/raw` (via `scripts/fetch_chain.py`) and watch the
  cleaning report change with market conditions.